# 2.2 數值資料正規化與標準化

這份 Notebook 比較 MinMaxScaler、StandardScaler 與 Keras Normalization layer。

## 1. 載入套件與建立資料

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

n = 800
x1 = rng.normal(50, 10, n)
x2 = rng.normal(5000, 1200, n)
x3 = rng.uniform(0, 1, n)
logit = 0.06 * (x1 - 50) + 0.001 * (x2 - 5000) + 2.0 * (x3 - 0.5)
y = (logit + rng.normal(0, 0.8, n) > 0).astype('int64')
X = np.column_stack([x1, x2, x3]).astype('float32')

pd.DataFrame(X, columns=['temperature', 'pressure', 'ratio']).describe().round(2)

## 2. 切分資料

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## 3. Min-max normalization

In [ ]:
minmax = MinMaxScaler()
x_train_minmax = minmax.fit_transform(x_train)
x_test_minmax = minmax.transform(x_test)

pd.DataFrame(x_train_minmax, columns=['temperature', 'pressure', 'ratio']).describe().round(3)

## 4. Standardization

In [ ]:
standard = StandardScaler()
x_train_std = standard.fit_transform(x_train)
x_test_std = standard.transform(x_test)

pd.DataFrame(x_train_std, columns=['temperature', 'pressure', 'ratio']).describe().round(3)

## 5. Keras Normalization layer

In [ ]:
normalizer = tf.keras.layers.Normalization()
normalizer.adapt(x_train)

normalized_preview = normalizer(x_train[:5]).numpy()
pd.DataFrame(normalized_preview, columns=['temperature', 'pressure', 'ratio']).round(3)

## 6. 將 Normalization 放進模型

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    normalizer,
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=25, batch_size=32, verbose=0)

pd.DataFrame([
    model.evaluate(x_train, y_train, verbose=0, return_dict=True),
    model.evaluate(x_test, y_test, verbose=0, return_dict=True),
], index=['train', 'test'])

## 7. 小結

需要估計統計量的前處理只 fit 訓練集。Keras Normalization layer 可以把數值前處理一起放進模型。